# RadGraph + BiomedCLIP + GraphRAG Knowledge Graph for Chest X-Ray Studies

This notebook builds a heterogeneous knowledge graph from a subset of paired chest X-ray images and JSON annotations. It combines:

- **RadGraph** report extraction: observation/anatomy entities and relations from radiology text.
- **BiomedCLIP** image/text embeddings: image nodes, label nodes, and concept-node features in a shared semantic space.
- **FAISS retrieval**: image-to-image `similar_to` edges from BiomedCLIP embeddings.
- **GraphRAG-style graph construction**: corpus-level entity resolution, graph materialization, and query-time subgraph retrieval.

For 10-fold cross validation, this notebook reads the existing `multi_kg_image_json_10fold.csv` split table and can generate one train-only KG file per fold. For quick smoke tests, set `MAX_IMAGES = 200`; for the full ~44k-image dataset, keep `MAX_IMAGES = None`. The graph treats multi-label classification as link prediction between `image` nodes and 14 fixed `label` nodes: `normal` plus the 13 CheXpert findings. Observed training/study labels become `image -> has_finding -> label` edges. RadGraph absent observations become separate `absent_mentions` edges and are not used as positive findings.

Primary references used for API choices:

- RadGraph PyPI: https://pypi.org/project/radgraph/
- GraphRAG PyPI: https://pypi.org/project/graphrag/

For each fold, the KG is built from rows marked `train` in that fold column. Rows marked `test` are saved separately as query/evaluation metadata and are not included in the fold training graph. The fold-specific `.pt` artifacts are named `kg_heterodata_fold{i}.pt`.


In [1]:
# Optional dependency installation. Set to True in a fresh environment.
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys
    packages = [
        "radgraph", "graphrag", "open_clip_torch", "transformers", "faiss-cpu",
        "networkx", "pandas", "numpy", "pillow", "tqdm", "scikit-learn", "torch_geometric",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])

In [2]:
import json
import os
import random
import re
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
try:
    from IPython.display import display
except ImportError:
    display = print

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Cloud paths used in your project. The local sample path is used only if the cloud paths are unavailable.
DATA_DIRS = [
    Path("/data/liangz2/openi/multi_kg/train"),
    Path("/data/liangz2/openi/multi_kg/test"),
]
LOCAL_SAMPLE_DIR = Path("/Users/liangz2/Documents/inverse_diffusion/data")

SPLIT_CSV_CANDIDATES = [
    Path("/data/liangz2/openi/multi_kg/multi_kg_image_json_10fold.csv"),
    Path("/data/liangz2/openi/inverse_diff/multi_kg_image_json_10fold.csv"),
    Path("/Users/liangz2/Documents/inverse_diffusion/multi_kg_image_json_10fold.csv"),
]
USE_10FOLD_SPLIT_CSV = True
BUILD_10FOLD_KG_FILES = True
FOLD_INDICES = list(range(10))
REUSE_EXISTING_FOLD_KG = True

BASE_OUTPUT = Path("/data/liangz2/openi/multi_kg") if Path("/data/liangz2/openi/multi_kg").exists() else Path.cwd()
OUTPUT_ROOT = BASE_OUTPUT
METADATA_DIR = OUTPUT_ROOT / "kg_metadata"
EMBEDDING_DIR = OUTPUT_ROOT / "kg_embeddings"
RADGRAPH_DIR = OUTPUT_ROOT / "kg_radgraph"
GRAPH_TABLE_DIR = OUTPUT_ROOT / "kg_graph_tables"
NODE_FEATURE_DIR = OUTPUT_ROOT / "kg_node_features"
NETWORKX_DIR = OUTPUT_ROOT / "kg_networkx"
QUERY_SUBGRAPH_DIR = OUTPUT_ROOT / "kg_query_subgraphs"
FOLD_KG_DIR = OUTPUT_ROOT / "kg_10fold"
PYG_DIR = OUTPUT_ROOT / "kg_pyg"
for directory in [METADATA_DIR, EMBEDDING_DIR, RADGRAPH_DIR, GRAPH_TABLE_DIR, NODE_FEATURE_DIR, NETWORKX_DIR, QUERY_SUBGRAPH_DIR, FOLD_KG_DIR, PYG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MAX_IMAGES = None  # None = full split CSV; set 200 for a quick smoke test
SHUFFLE_INPUTS = False
REPORT_TEXT_SOURCE = "clinical"  # "clinical" avoids bbox-generated text; set "report" to force JSON report field.

BIOMEDCLIP_REPO = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
BATCH_SIZE = 32
NUM_WORKERS = 0
NORMALIZE_EMBEDDINGS = True

RUN_RADGRAPH = True
RADGRAPH_MODEL = "modern-radgraph-xl"
RADGRAPH_BATCH_SIZE = 8
USE_CACHED_RADGRAPH = True

KNN_K = 20
QUERY_TOP_K = 10

print("DEVICE", DEVICE, "DTYPE", DTYPE)
print("OUTPUT_ROOT", OUTPUT_ROOT)
print("category output dirs", {
    "metadata": METADATA_DIR,
    "embeddings": EMBEDDING_DIR,
    "radgraph": RADGRAPH_DIR,
    "graph_tables": GRAPH_TABLE_DIR,
    "node_features": NODE_FEATURE_DIR,
    "networkx": NETWORKX_DIR,
    "query_subgraphs": QUERY_SUBGRAPH_DIR,
    "fold_kg": FOLD_KG_DIR,
    "pyg": PYG_DIR,
})


DEVICE cuda DTYPE torch.float16
OUTPUT_ROOT /data/liangz2/openi/multi_kg
category output dirs {'metadata': PosixPath('/data/liangz2/openi/multi_kg/kg_metadata'), 'embeddings': PosixPath('/data/liangz2/openi/multi_kg/kg_embeddings'), 'radgraph': PosixPath('/data/liangz2/openi/multi_kg/kg_radgraph'), 'graph_tables': PosixPath('/data/liangz2/openi/multi_kg/kg_graph_tables'), 'node_features': PosixPath('/data/liangz2/openi/multi_kg/kg_node_features'), 'networkx': PosixPath('/data/liangz2/openi/multi_kg/kg_networkx'), 'query_subgraphs': PosixPath('/data/liangz2/openi/multi_kg/kg_query_subgraphs'), 'fold_kg': PosixPath('/data/liangz2/openi/multi_kg/kg_10fold'), 'pyg': PosixPath('/data/liangz2/openi/multi_kg/kg_pyg')}


## Label Schema and Text Normalization

The graph uses 14 shared label nodes. Positive disease edges come from the JSON `labels` field, while the `normal` edge comes from `normal == "yes"`. RadGraph absent mentions are intentionally kept separate so phrases like "no pneumothorax" do not become positive pneumothorax evidence.


In [3]:
FINDING_LABELS = (
    "enlarged cardiomediastinum", "cardiomegaly", "lung opacity", "lung lesion", "edema",
    "consolidation", "pneumonia", "atelectasis", "pneumothorax",
    "pleural effusion", "pleural other", "fracture", "support devices",
)
SCORING_LABELS = ("normal",) + FINDING_LABELS
TARGET_COLUMNS = ["y_" + label.replace(" ", "_") for label in SCORING_LABELS]

LABEL_ALIASES = {
    "enlarged cardiomediastinum": "enlarged cardiomediastinum",
    "cardiomegaly": "cardiomegaly",
    "lung opacity": "lung opacity",
    "lung lesion": "lung lesion",
    "edema": "edema",
    "consolidation": "consolidation",
    "pneumonia": "pneumonia",
    "atelectasis": "atelectasis",
    "pneumothorax": "pneumothorax",
    "pleural effusion": "pleural effusion",
    "pleural other": "pleural other",
    "fracture": "fracture",
    "support devices": "support devices",
    "support device": "support devices",
}

CHEXPERT_CONCEPT_PATTERNS = {
    "enlarged cardiomediastinum": [
        r"enlarged cardiomediastinum", r"cardiomediastinal enlargement", r"widened mediastinum",
        r"mediastinal widening", r"enlarged mediastinal silhouette",
    ],
    "cardiomegaly": [
        r"cardiomegaly", r"enlarged heart", r"cardiac silhouette enlarged", r"heart enlarged",
        r"enlargement of .* heart", r"cardiac enlargement",
    ],
    "lung opacity": [
        r"opacity", r"opacities", r"airspace disease", r"airspace opacity", r"infiltrate", r"hazy opacity",
        r"reticular opacity", r"density", r"increased markings",
    ],
    "lung lesion": [
        r"lung lesion", r"pulmonary nodule", r"nodule", r"mass", r"cavitary lesion", r"lesion",
    ],
    "edema": [
        r"edema", r"pulmonary edema", r"vascular congestion", r"interstitial edema", r"fluid overload",
    ],
    "consolidation": [
        r"consolidation", r"consolidative", r"airspace consolidation",
    ],
    "pneumonia": [
        r"pneumonia", r"infectious process", r"infection", r"pneumonic",
    ],
    "atelectasis": [
        r"atelectasis", r"atelectatic", r"volume loss", r"subsegmental atelectasis", r"linear atelectasis",
    ],
    "pneumothorax": [
        r"pneumothorax", r"apical pneumothorax", r"ptx",
    ],
    "pleural effusion": [
        r"pleural effusion", r"effusion", r"pleural fluid", r"fluid collection",
    ],
    "pleural other": [
        r"pleural thickening", r"pleural plaque", r"pleural scarring", r"pleural calcification",
    ],
    "fracture": [
        r"fracture", r"rib fracture", r"clavicle fracture", r"vertebral compression", r"osseous fracture",
    ],
    "support devices": [
        r"support device", r"tube", r"line", r"catheter", r"picc", r"port", r"pacemaker",
        r"endotracheal", r"enteric", r"chest tube", r"central venous", r"hardware",
    ],
}

STOP_TOKENS = {
    "a", "an", "the", "of", "and", "or", "with", "without", "to", "in", "on", "at", "for", "from",
    "there", "is", "are", "was", "were", "seen", "noted", "present", "evidence", "no", "acute",
}


def normalize_space(text: str) -> str:
    text = str(text or "").replace("\n", " ").replace("_", " ").replace("-", " ")
    text = re.sub(r"[^A-Za-z0-9/ ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def normalize_label(label: str) -> Optional[str]:
    key = normalize_space(label)
    return LABEL_ALIASES.get(key)


def canonicalize_concept(text: str) -> str:
    text = normalize_space(text)
    toks = [tok for tok in text.split() if tok not in STOP_TOKENS]
    return " ".join(toks) if toks else text


def labels_from_json(record: Dict) -> List[str]:
    raw = record.get("labels", []) or record.get("label", []) or []
    if isinstance(raw, str):
        raw = [raw]
    out = []
    for item in raw:
        label = normalize_label(item)
        if label and label not in out:
            out.append(label)
    return out


def concept_maps_to_labels(concept: str) -> List[str]:
    concept_norm = normalize_space(concept)
    matches = []
    for label, patterns in CHEXPERT_CONCEPT_PATTERNS.items():
        if any(re.search(pattern, concept_norm) for pattern in patterns):
            matches.append(label)
    return matches


def is_normal_record(record: Dict) -> bool:
    return normalize_space(record.get("normal", "")) == "yes"


## Load Image/JSON Pairs

The notebook scans the cloud train/test folders first. If they are unavailable in the current environment, it falls back to the local sample folder under the project directory.


In [4]:
def available_data_dirs() -> List[Path]:
    dirs = [p for p in DATA_DIRS if p.exists()]
    if dirs:
        return dirs
    if LOCAL_SAMPLE_DIR.exists():
        print("Cloud data folders not found. Using local sample folder:", LOCAL_SAMPLE_DIR)
        return [LOCAL_SAMPLE_DIR]
    raise FileNotFoundError("No data folders found. Check DATA_DIRS or LOCAL_SAMPLE_DIR.")


def find_split_csv() -> Optional[Path]:
    for path in SPLIT_CSV_CANDIDATES:
        if path.exists():
            return path
    return None


def paired_image_for_json(json_path: Path) -> Optional[Path]:
    for suffix in (".jpg", ".jpeg", ".png"):
        image_path = json_path.with_suffix(suffix)
        if image_path.exists():
            return image_path
    return None


def resolve_existing_path(path_value, suffix: Optional[str] = None) -> Optional[Path]:
    if path_value is None:
        return None
    path_text = str(path_value)
    if not path_text or path_text.lower() == "nan":
        return None
    path = Path(path_text)
    if suffix is not None:
        path = path.with_suffix(suffix)
    if path.exists():
        return path
    replacements = [
        ("/vf/users/liangz2/openi/multi_kg", "/data/liangz2/openi/multi_kg"),
        ("/data/liangz2/openi/multi_kg", "/vf/users/liangz2/openi/multi_kg"),
    ]
    for old, new in replacements:
        if path_text.startswith(old):
            candidate = Path(path_text.replace(old, new, 1))
            if suffix is not None:
                candidate = candidate.with_suffix(suffix)
            if candidate.exists():
                return candidate
    basename = path.with_suffix(suffix).name if suffix is not None else path.name
    for data_dir in DATA_DIRS:
        candidate = data_dir / basename
        if candidate.exists():
            return candidate
    if LOCAL_SAMPLE_DIR.exists():
        candidate = LOCAL_SAMPLE_DIR / basename
        if candidate.exists():
            return candidate
    return path


def value_present(value) -> bool:
    if value is None:
        return False
    if isinstance(value, float) and pd.isna(value):
        return False
    if isinstance(value, str) and not value.strip():
        return False
    return True


def parse_label_list(value) -> List[str]:
    if not value_present(value):
        return []
    if isinstance(value, list):
        raw = value
    else:
        text = str(value).strip()
        try:
            parsed = json.loads(text)
            raw = parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            raw = [part.strip() for part in re.split(r"[,;|]", text) if part.strip()]
    out = []
    for item in raw:
        label = normalize_label(item)
        if label and label not in out:
            out.append(label)
    return out


def extract_report_text(record: Dict) -> str:
    if REPORT_TEXT_SOURCE == "report":
        return str(record.get("report", "") or record.get("Original_Caption", "") or "").strip()
    clinical_parts = []
    for key in ("INDICATION", "FINDINGS", "IMPRESSION"):
        value = str(record.get(key, "") or "").strip()
        if value:
            clinical_parts.append(f"{key.title()}: {value}")
    if clinical_parts:
        return " ".join(clinical_parts)
    for key in ("Original_Caption", "Caption", "description", "report"):
        value = str(record.get(key, "") or "").strip()
        if value:
            return value
    return ""


def read_json_or_empty(json_path: Optional[Path]) -> Dict:
    if json_path is None or not Path(json_path).exists():
        return {}
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def row_to_metadata(image_path: Path, json_path: Optional[Path], record: Dict, split_row: Optional[pd.Series] = None) -> Dict:
    split_row = split_row if split_row is not None else pd.Series(dtype=object)
    labels = labels_from_json(record) if record else []
    if not labels:
        for key in ("labels_normalized", "labels", "label", "labels_raw"):
            if key in split_row and value_present(split_row.get(key)):
                labels = parse_label_list(split_row.get(key))
                break
    if record:
        normal = is_normal_record(record)
    else:
        normal = normalize_space(split_row.get("normal", "")) == "yes"
    study_id = str(record.get("study_id") or split_row.get("study_id", "") or image_path.stem)
    row = {
        "study_id": study_id,
        "subject_id": str(record.get("subject_id") or split_row.get("subject_id", "")),
        "image_path": str(image_path),
        "json_path": str(json_path or image_path.with_suffix(".json")),
        "image_name": str(record.get("image_name") or split_row.get("image_name", "") or image_path.name),
        "view_position": str(record.get("view_position") or split_row.get("view_position", "")),
        "normal": "yes" if normal else "no",
        "labels_raw": json.dumps(record.get("labels", record.get("label", []))) if record else json.dumps(labels),
        "labels_normalized": json.dumps(labels),
        "report_text": extract_report_text(record) if record else str(split_row.get("report_text", "") or split_row.get("report", "") or ""),
    }
    for label, col in zip(SCORING_LABELS, TARGET_COLUMNS):
        row[col] = int(normal) if label == "normal" else int(label in labels)
    return row


def load_subset_from_split_csv(split_csv: Path, max_images: Optional[int] = MAX_IMAGES) -> pd.DataFrame:
    split_df = pd.read_csv(split_csv)
    if max_images is not None:
        split_df = split_df.head(int(max_images)).copy()
    rows, missing = [], []
    for _, split_row in tqdm(split_df.iterrows(), total=len(split_df), desc="read split csv image/json rows"):
        image_source = split_row.get("image_path", split_row.get("path", split_row.get("image", "")))
        image_path = resolve_existing_path(image_source)
        if image_path is None or not image_path.exists():
            missing.append({"image_path": image_source, "study_id": split_row.get("study_id", "")})
            continue
        json_source = split_row.get("json_path", "")
        json_path = resolve_existing_path(json_source) if value_present(json_source) else resolve_existing_path(image_path, suffix=".json")
        record = read_json_or_empty(json_path)
        row = row_to_metadata(image_path, json_path, record, split_row=split_row)
        for col in split_df.columns:
            if col.startswith("fold_"):
                row[col] = str(split_row[col]).strip().lower()
        rows.append(row)
    if missing:
        missing_path = METADATA_DIR / "missing_split_paths.csv"
        pd.DataFrame(missing).to_csv(missing_path, index=False)
        print(f"Warning: {len(missing)} split rows had missing image paths. Saved {missing_path}")
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No records could be loaded from split CSV.")
    df["split_csv"] = str(split_csv)
    return df.reset_index(drop=True)


def load_subset_from_folders(max_images: Optional[int] = MAX_IMAGES) -> pd.DataFrame:
    pairs = []
    for data_dir in available_data_dirs():
        for json_path in sorted(data_dir.glob("*.json")):
            image_path = paired_image_for_json(json_path)
            if image_path is not None:
                pairs.append((image_path, json_path))
    if SHUFFLE_INPUTS:
        rng = random.Random(SEED)
        rng.shuffle(pairs)
    if max_images is not None:
        pairs = pairs[:int(max_images)]
    rows = []
    for image_path, json_path in tqdm(pairs, desc="read json pairs"):
        record = read_json_or_empty(json_path)
        rows.append(row_to_metadata(image_path, json_path, record))
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No paired image/JSON records were found.")
    return df.reset_index(drop=True)


def load_subset(max_images: Optional[int] = MAX_IMAGES) -> pd.DataFrame:
    split_csv = find_split_csv() if USE_10FOLD_SPLIT_CSV else None
    if split_csv is not None:
        print("Using split CSV:", split_csv)
        return load_subset_from_split_csv(split_csv, max_images=max_images)
    return load_subset_from_folders(max_images=max_images)

studies_df = load_subset(MAX_IMAGES)
studies_df["global_row_pos"] = np.arange(len(studies_df), dtype=np.int64)
studies_df.to_csv(METADATA_DIR / "study_subset_metadata.csv", index=False)
print("studies", len(studies_df))
fold_cols = [col for col in studies_df.columns if col.startswith("fold_")]
print("fold columns", fold_cols[:5], "...", len(fold_cols))
display(studies_df[["study_id", "subject_id", "image_path", "normal", "labels_normalized", "report_text"] + fold_cols[:1]].head())


Using split CSV: /data/liangz2/openi/multi_kg/multi_kg_image_json_10fold.csv


read split csv image/json rows:   0%|          | 0/44100 [00:00<?, ?it/s]

studies 44100
fold columns ['fold_id', 'fold_0', 'fold_1', 'fold_2', 'fold_3'] ... 11


,study_id,subject_id,image_path,normal,labels_normalized,report_text,fold_id
0,50003542,18703657,/vf/users/liangz2/openi/multi_kg/train/5000354...,yes,[],Indication: ___-year-old female with chest pai...,0
1,50003755,18699864,/vf/users/liangz2/openi/multi_kg/train/5000375...,yes,[],"Indication: ___M with PTX, CT removed today. F...",0
2,50008255,15749643,/vf/users/liangz2/openi/multi_kg/train/5000825...,yes,[],Indication: Altered mental status and syncope....,0
3,50010166,18092696,/vf/users/liangz2/openi/multi_kg/train/5001016...,no,"[""lung opacity"", ""pneumonia""]",Indication: ___-year-old woman with cough and ...,0
4,50010229,13446842,/vf/users/liangz2/openi/multi_kg/train/5001022...,no,"[""atelectasis"", ""cardiomegaly"", ""consolidation...",Indication: History: ___M with COPD and CHF p/...,0


## Encode Images and Text with BiomedCLIP

Image nodes use BiomedCLIP image embeddings. Label, observation, and anatomy nodes use BiomedCLIP text embeddings, which keeps node features in a related semantic space.


In [5]:
try:
    import open_clip
    from open_clip import create_model_from_pretrained, get_tokenizer
except ImportError as exc:
    raise ImportError("Install open_clip_torch first, or set INSTALL_DEPS=True in the first code cell.") from exc

biomedclip, biomedclip_preprocess = create_model_from_pretrained(BIOMEDCLIP_REPO)
biomedclip_tokenizer = get_tokenizer(BIOMEDCLIP_REPO)
biomedclip = biomedclip.to(DEVICE, dtype=DTYPE).eval()
for param in biomedclip.parameters():
    param.requires_grad_(False)

class ImagePathDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, preprocess):
        self.frame = frame.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int):
        image_path = self.frame.loc[idx, "image_path"]
        with Image.open(image_path) as image:
            tensor = self.preprocess(image.convert("RGB"))
        return tensor, idx

@torch.no_grad()
def encode_images(frame: pd.DataFrame) -> np.ndarray:
    dataset = ImagePathDataset(frame, biomedclip_preprocess)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    features, indices = [], []
    for images, idx in tqdm(loader, desc="BiomedCLIP image encode"):
        images = images.to(DEVICE, dtype=DTYPE)
        feats = biomedclip.encode_image(images).float()
        if NORMALIZE_EMBEDDINGS:
            feats = F.normalize(feats, dim=-1)
        features.append(feats.cpu().numpy().astype("float32"))
        indices.append(idx.numpy())
    embeddings = np.concatenate(features, axis=0)
    order = np.concatenate(indices, axis=0)
    if not np.array_equal(order, np.arange(len(frame))):
        raise RuntimeError("Image embedding order mismatch.")
    return np.ascontiguousarray(embeddings.astype("float32"))

@torch.no_grad()
def encode_texts(texts: Sequence[str], batch_size: int = 64) -> np.ndarray:
    features = []
    for start in tqdm(range(0, len(texts), batch_size), desc="BiomedCLIP text encode"):
        batch = list(texts[start:start + batch_size])
        tokens = biomedclip_tokenizer(batch, context_length=256).to(DEVICE)
        feats = biomedclip.encode_text(tokens).float()
        if NORMALIZE_EMBEDDINGS:
            feats = F.normalize(feats, dim=-1)
        features.append(feats.cpu().numpy().astype("float32"))
    return np.ascontiguousarray(np.concatenate(features, axis=0).astype("float32")) if texts else np.empty((0, 512), dtype="float32")

image_embeddings_path = EMBEDDING_DIR / "image_embeddings.npy"
if image_embeddings_path.exists() and len(np.load(image_embeddings_path, mmap_mode="r")) == len(studies_df):
    image_embeddings = np.load(image_embeddings_path).astype("float32")
    print("Loaded cached image embeddings:", image_embeddings.shape)
else:
    image_embeddings = encode_images(studies_df)
    np.save(image_embeddings_path, image_embeddings)
    print("Saved image embeddings:", image_embeddings.shape)


BiomedCLIP image encode:   0%|          | 0/1379 [00:00<?, ?it/s]

Saved image embeddings: (44100, 512)


## Run RadGraph and Cache Report Annotations

RadGraph returns entity spans and relations per report. The raw output is cached so graph construction can be rerun without re-extracting reports.


In [5]:
radgraph_cache_path = RADGRAPH_DIR / "radgraph_raw_annotations.json"


def _find_wrapped_tokenizer_with_method(obj, method_name: str):
    """Find the real Hugging Face tokenizer inside an AllenNLP TokenizersBackend wrapper."""
    for attr in ("tokenizer", "_tokenizer", "hf_tokenizer", "_hf_tokenizer"):
        try:
            tokenizer = object.__getattribute__(obj, attr)
        except Exception:
            continue
        if hasattr(tokenizer, method_name):
            return tokenizer

    # Some wrappers store the backend tokenizer under less obvious attributes.
    try:
        values = vars(obj).values()
    except Exception:
        values = []
    for value in values:
        if hasattr(value, method_name):
            return value

    # NamedTuple-like wrappers.
    try:
        values = obj._asdict().values()
    except Exception:
        values = []
    for value in values:
        if hasattr(value, method_name):
            return value
    return None


def _special_token_id(obj, attr_name: str, default: int) -> int:
    for source in [obj, _find_wrapped_tokenizer_with_method(obj, attr_name)]:
        if source is None:
            continue
        try:
            value = getattr(source, attr_name)
        except Exception:
            continue
        if value is not None:
            return int(value)
    return int(default)


def _bert_special_token_fallback(method_name: str, self, *args, **kwargs):
    """BERT-compatible fallback for AllenNLP TokenizersBackend.

    RadGraph modern-radgraph-xl is BERT-based. If AllenNLP's TokenizersBackend
    hides the wrapped tokenizer, these helpers reproduce the standard BERT
    special-token behavior: [CLS] sequence [SEP] sequence_pair [SEP].
    """
    cls_id = _special_token_id(self, "cls_token_id", 101)
    sep_id = _special_token_id(self, "sep_token_id", 102)

    if method_name == "build_inputs_with_special_tokens":
        token_ids_0 = list(args[0]) if len(args) >= 1 else list(kwargs.get("token_ids_0", []))
        token_ids_1 = None
        if len(args) >= 2:
            token_ids_1 = args[1]
        elif "token_ids_1" in kwargs:
            token_ids_1 = kwargs["token_ids_1"]
        if token_ids_1 is None:
            return [cls_id] + token_ids_0 + [sep_id]
        return [cls_id] + token_ids_0 + [sep_id] + list(token_ids_1) + [sep_id]

    if method_name == "num_special_tokens_to_add":
        pair = bool(args[0]) if args else bool(kwargs.get("pair", False))
        return 3 if pair else 2

    if method_name == "get_special_tokens_mask":
        token_ids_0 = list(args[0]) if len(args) >= 1 else list(kwargs.get("token_ids_0", []))
        token_ids_1 = None
        if len(args) >= 2:
            token_ids_1 = args[1]
        elif "token_ids_1" in kwargs:
            token_ids_1 = kwargs["token_ids_1"]
        already_has_special_tokens = bool(kwargs.get("already_has_special_tokens", False))
        if already_has_special_tokens:
            return [1 if tok in {cls_id, sep_id} else 0 for tok in token_ids_0]
        if token_ids_1 is None:
            return [1] + [0] * len(token_ids_0) + [1]
        return [1] + [0] * len(token_ids_0) + [1] + [0] * len(token_ids_1) + [1]

    if method_name == "create_token_type_ids_from_sequences":
        token_ids_0 = list(args[0]) if len(args) >= 1 else list(kwargs.get("token_ids_0", []))
        token_ids_1 = None
        if len(args) >= 2:
            token_ids_1 = args[1]
        elif "token_ids_1" in kwargs:
            token_ids_1 = kwargs["token_ids_1"]
        if token_ids_1 is None:
            return [0] * (len(token_ids_0) + 2)
        return [0] * (len(token_ids_0) + 2) + [1] * (len(token_ids_1) + 1)

    raise AttributeError(
        f"{self.__class__.__name__} has no wrapped tokenizer method {method_name}"
    )


def _forward_tokenizer_method(method_name: str):
    def method(self, *args, **kwargs):
        tokenizer = _find_wrapped_tokenizer_with_method(self, method_name)
        if tokenizer is not None:
            return getattr(tokenizer, method_name)(*args, **kwargs)
        if method_name == "encode_plus" and callable(self):
            return self(*args, **kwargs)
        return _bert_special_token_fallback(method_name, self, *args, **kwargs)
    return method


def _patch_class_method(cls, method_name: str, patched: List[str], name: str) -> None:
    if cls is not None and not hasattr(cls, method_name):
        setattr(cls, method_name, _forward_tokenizer_method(method_name))
        patched.append(f"{name}.{method_name}")


def patch_radgraph_tokenizer_compatibility() -> None:
    """Patch tokenizer methods expected by RadGraph/AllenNLP.

    Some RadGraph archives are loaded through AllenNLP, whose tokenizer wrapper may
    miss helper methods that newer Hugging Face tokenizers still expose on the real
    tokenizer object. We forward those calls to the wrapped tokenizer.
    """
    patched = []
    tokenizer_methods = [
        "encode_plus",
        "build_inputs_with_special_tokens",
        "num_special_tokens_to_add",
        "get_special_tokens_mask",
        "create_token_type_ids_from_sequences",
        "convert_tokens_to_ids",
        "convert_ids_to_tokens",
        "tokenize",
        "decode",
    ]

    try:
        import transformers
        for class_name in [
            "PreTrainedTokenizer", "PreTrainedTokenizerFast",
            "BertTokenizer", "BertTokenizerFast",
            "AutoTokenizer",
        ]:
            cls = getattr(transformers, class_name, None)
            for method_name in tokenizer_methods:
                _patch_class_method(cls, method_name, patched, f"transformers.{class_name}")
        print("transformers version:", getattr(transformers, "__version__", "unknown"))
    except ImportError:
        pass

    backend_module_names = [
        "allennlp.common.cached_transformers",
        "allennlp.data.tokenizers.pretrained_transformer_tokenizer",
        "allennlp.data.tokenizers.pretrained_transformer_mismatched_tokenizer",
    ]
    for module_name in backend_module_names:
        try:
            module = __import__(module_name, fromlist=["TokenizersBackend"])
        except Exception:
            continue
        cls = getattr(module, "TokenizersBackend", None)
        for method_name in tokenizer_methods:
            _patch_class_method(cls, method_name, patched, f"{module_name}.TokenizersBackend")

    if patched:
        print("Patched tokenizer compatibility for RadGraph:", patched)
    else:
        print("No tokenizer compatibility patches were needed.")

def load_radgraph_model(model_name: str):
    patch_radgraph_tokenizer_compatibility()
    from radgraph import RadGraph
    try:
        return RadGraph(model_type=model_name)
    except TypeError:
        return RadGraph(model=model_name)


def get_radgraph_item(output: Dict, local_index: int) -> Dict:
    for key in (str(local_index), local_index):
        if key in output:
            return output[key]
    # Last resort: preserve order if the package returns a plain dict with unexpected keys.
    values = list(output.values()) if isinstance(output, dict) else []
    return values[local_index] if local_index < len(values) else {}


def run_radgraph_on_reports(frame: pd.DataFrame) -> Dict[str, Dict]:
    if USE_CACHED_RADGRAPH and radgraph_cache_path.exists():
        with open(radgraph_cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        if set(cached.keys()) >= set(frame["study_id"].astype(str)):
            print("Loaded cached RadGraph annotations:", radgraph_cache_path)
            return {sid: cached[sid] for sid in frame["study_id"].astype(str)}
    if not RUN_RADGRAPH:
        print("RUN_RADGRAPH=False; returning empty annotation map.")
        return {}

    radgraph = load_radgraph_model(RADGRAPH_MODEL)
    annotations_by_study = {}
    reports = frame["report_text"].fillna("").astype(str).tolist()
    study_ids = frame["study_id"].astype(str).tolist()
    for start in tqdm(range(0, len(reports), RADGRAPH_BATCH_SIZE), desc="RadGraph"):
        batch_reports = reports[start:start + RADGRAPH_BATCH_SIZE]
        batch_ids = study_ids[start:start + RADGRAPH_BATCH_SIZE]
        output = radgraph(batch_reports)
        for local_index, study_id in enumerate(batch_ids):
            annotations_by_study[study_id] = get_radgraph_item(output, local_index)
    with open(radgraph_cache_path, "w", encoding="utf-8") as f:
        json.dump(annotations_by_study, f, indent=2)
    return annotations_by_study

radgraph_annotations = run_radgraph_on_reports(studies_df)
print("RadGraph annotated studies:", len(radgraph_annotations))
first_key = next(iter(radgraph_annotations), None)
if first_key:
    print(first_key, list(radgraph_annotations[first_key].keys()))

transformers version: 5.11.0
Patched tokenizer compatibility for RadGraph: ['transformers.PreTrainedTokenizer.encode_plus', 'transformers.PreTrainedTokenizerFast.encode_plus', 'transformers.PreTrainedTokenizerFast.build_inputs_with_special_tokens', 'transformers.PreTrainedTokenizerFast.create_token_type_ids_from_sequences', 'transformers.AutoTokenizer.encode_plus', 'transformers.AutoTokenizer.build_inputs_with_special_tokens', 'transformers.AutoTokenizer.num_special_tokens_to_add', 'transformers.AutoTokenizer.get_special_tokens_mask', 'transformers.AutoTokenizer.create_token_type_ids_from_sequences', 'transformers.AutoTokenizer.convert_tokens_to_ids', 'transformers.AutoTokenizer.convert_ids_to_tokens', 'transformers.AutoTokenizer.tokenize', 'transformers.AutoTokenizer.decode']
Using device: cuda:0


RadGraph:   0%|          | 0/5513 [00:00<?, ?it/s]

RadGraph annotated studies: 44100
50003542 ['text', 'entities', 'data_source', 'data_split']


## Convert RadGraph Output to a Corpus-Level KG

This section canonicalizes RadGraph spans, creates global observation/anatomy nodes, links observations to CheXpert label nodes, and materializes typed edge tables.


In [6]:
def split_radgraph_label(label: str) -> Tuple[str, str]:
    label = str(label or "")
    if "::" in label:
        kind, status = label.split("::", 1)
    else:
        parts = label.split()
        kind, status = (parts[0] if parts else "Unknown"), " ".join(parts[1:])
    return normalize_space(kind), normalize_space(status)


def node_id(node_type: str, name: str) -> str:
    return f"{node_type}::{canonicalize_concept(name)}"


def relation_edge_type(relation_name: str) -> str:
    return normalize_space(relation_name).replace(" ", "_")


def build_kg_tables(frame: pd.DataFrame, annotations_by_study: Dict[str, Dict]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    node_rows = {}
    edge_rows = []

    # Shared label nodes.
    for label in SCORING_LABELS:
        nid = f"label::{label}"
        node_rows[nid] = {"node_id": nid, "node_type": "label", "name": label, "canonical_name": label}

    # Image nodes and JSON label supervision edges.
    for row_pos, row in frame.reset_index(drop=True).iterrows():
        image_id = f"image::{row['study_id']}"
        node_rows[image_id] = {
            "node_id": image_id,
            "node_type": "image",
            "name": str(row["study_id"]),
            "canonical_name": str(row["study_id"]),
            "study_id": str(row["study_id"]),
            "subject_id": str(row.get("subject_id", "")),
            "image_path": str(row["image_path"]),
            "row_pos": int(row_pos),
        }
        for label, col in zip(SCORING_LABELS, TARGET_COLUMNS):
            if int(row[col]) == 1:
                edge_rows.append({
                    "src": image_id, "dst": f"label::{label}", "edge_type": "has_finding",
                    "src_type": "image", "dst_type": "label", "label": label, "weight": 1.0,
                    "study_id": str(row["study_id"]), "source": "json_labels",
                })

    # RadGraph nodes and edges.
    for _, row in frame.iterrows():
        study_id = str(row["study_id"])
        image_id = f"image::{study_id}"
        ann = annotations_by_study.get(study_id, {}) or {}
        entities = ann.get("entities", {}) or {}
        entity_to_node = {}
        entity_meta = {}

        for ent_id, ent in entities.items():
            tokens = str(ent.get("tokens", "") or "").strip()
            if not tokens:
                continue
            kind, status = split_radgraph_label(ent.get("label", ""))
            canonical = canonicalize_concept(tokens)
            if kind.startswith("observation"):
                ntype = "observation"
                nid = node_id("observation", canonical)
            elif kind.startswith("anatomy"):
                ntype = "anatomy"
                nid = node_id("anatomy", canonical)
            else:
                ntype = "radgraph_entity"
                nid = node_id("entity", canonical)
            node_rows.setdefault(nid, {
                "node_id": nid, "node_type": ntype, "name": tokens, "canonical_name": canonical,
            })
            entity_to_node[str(ent_id)] = nid
            entity_meta[str(ent_id)] = {"kind": kind, "status": status, "tokens": tokens, "canonical": canonical, "node_id": nid}

            if ntype == "observation":
                if "absent" in status:
                    image_edge_type = "absent_mentions"
                elif "uncertain" in status:
                    image_edge_type = "uncertain_mentions"
                else:
                    image_edge_type = "mentions"
                edge_rows.append({
                    "src": image_id, "dst": nid, "edge_type": image_edge_type,
                    "src_type": "image", "dst_type": "observation", "label": "", "weight": 1.0,
                    "study_id": study_id, "source": "radgraph", "radgraph_status": status,
                    "surface_text": tokens,
                })
                for mapped_label in concept_maps_to_labels(canonical):
                    edge_rows.append({
                        "src": nid, "dst": f"label::{mapped_label}", "edge_type": "maps_to",
                        "src_type": "observation", "dst_type": "label", "label": mapped_label,
                        "weight": 1.0, "study_id": study_id, "source": "lexicon_bridge",
                        "radgraph_status": status, "surface_text": tokens,
                    })

        for ent_id, ent in entities.items():
            src_node = entity_to_node.get(str(ent_id))
            if src_node is None:
                continue
            src_meta = entity_meta.get(str(ent_id), {})
            for rel in ent.get("relations", []) or []:
                if not isinstance(rel, (list, tuple)) or len(rel) < 2:
                    continue
                rel_name, dst_ent_id = rel[0], str(rel[1])
                dst_node = entity_to_node.get(dst_ent_id)
                if dst_node is None:
                    continue
                dst_meta = entity_meta.get(dst_ent_id, {})
                edge_type = relation_edge_type(rel_name)
                edge_rows.append({
                    "src": src_node, "dst": dst_node, "edge_type": edge_type,
                    "src_type": node_rows[src_node]["node_type"], "dst_type": node_rows[dst_node]["node_type"],
                    "label": "", "weight": 1.0, "study_id": study_id, "source": "radgraph_relation",
                    "radgraph_status": src_meta.get("status", ""),
                    "surface_text": src_meta.get("tokens", ""),
                    "dst_surface_text": dst_meta.get("tokens", ""),
                })

    nodes_df = pd.DataFrame(node_rows.values()).fillna("")
    edges_df = pd.DataFrame(edge_rows).fillna("")
    return nodes_df, edges_df

nodes_df, edges_df = build_kg_tables(studies_df, radgraph_annotations)
nodes_df.to_csv(GRAPH_TABLE_DIR / "kg_nodes.csv", index=False)
edges_df.to_csv(GRAPH_TABLE_DIR / "kg_edges_radgraph_and_labels.csv", index=False)
print("nodes", nodes_df.shape, nodes_df["node_type"].value_counts().to_dict())
print("edges", edges_df.shape, edges_df["edge_type"].value_counts().to_dict())
display(nodes_df.head())
display(edges_df.head())


nodes (49711, 8) {'image': 44100, 'observation': 3492, 'anatomy': 2105, 'label': 14}
edges (1267162, 12) {'mentions': 311735, 'absent_mentions': 266841, 'modify': 236228, 'located_at': 233874, 'maps_to': 124065, 'has_finding': 56474, 'uncertain_mentions': 20425, 'suggestive_of': 17520}


,node_id,node_type,name,canonical_name,study_id,subject_id,image_path,row_pos
0,label::normal,label,normal,normal,,,,
1,label::enlarged cardiomediastinum,label,enlarged cardiomediastinum,enlarged cardiomediastinum,,,,
2,label::cardiomegaly,label,cardiomegaly,cardiomegaly,,,,
3,label::lung opacity,label,lung opacity,lung opacity,,,,
4,label::lung lesion,label,lung lesion,lung lesion,,,,


,src,dst,edge_type,src_type,dst_type,label,weight,study_id,source,radgraph_status,surface_text,dst_surface_text
0,image::50003542,label::normal,has_finding,image,label,normal,1.0,50003542,json_labels,,,
1,image::50003755,label::normal,has_finding,image,label,normal,1.0,50003755,json_labels,,,
2,image::50008255,label::normal,has_finding,image,label,normal,1.0,50008255,json_labels,,,
3,image::50010166,label::lung opacity,has_finding,image,label,lung opacity,1.0,50010166,json_labels,,,
4,image::50010166,label::pneumonia,has_finding,image,label,pneumonia,1.0,50010166,json_labels,,,


## FAISS Similarity Edges

The FAISS index is built over the selected subset's BiomedCLIP image embeddings. For full experiments, build the index only on the training partition to avoid retrieval leakage.


In [8]:
try:
    import faiss
except ImportError as exc:
    raise ImportError("Install faiss-cpu or faiss-gpu first, or set INSTALL_DEPS=True.") from exc


def l2_normalize_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)).astype("float32")


def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    emb = l2_normalize_np(embeddings.astype("float32"))
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(np.ascontiguousarray(emb))
    return index


def materialize_similarity_edges(frame: pd.DataFrame, embeddings: np.ndarray, k: int = KNN_K) -> pd.DataFrame:
    index = build_faiss_index(embeddings)
    search_k = min(k + 1, len(frame))
    sims, idx = index.search(l2_normalize_np(embeddings), search_k)
    rows = []
    for i in range(len(frame)):
        src_study = str(frame.iloc[i]["study_id"])
        src_subject = str(frame.iloc[i].get("subject_id", ""))
        rank = 0
        for sim, j in zip(sims[i], idx[i]):
            j = int(j)
            if j < 0 or j == i:
                continue
            dst_study = str(frame.iloc[j]["study_id"])
            dst_subject = str(frame.iloc[j].get("subject_id", ""))
            rank += 1
            rows.append({
                "src": f"image::{src_study}", "dst": f"image::{dst_study}", "edge_type": "similar_to",
                "src_type": "image", "dst_type": "image", "weight": float(sim), "rank": int(rank),
                "study_id": src_study, "dst_study_id": dst_study, "source": "faiss_biomedclip",
                "same_subject": bool(src_subject and src_subject == dst_subject),
            })
            if rank >= k:
                break
    return pd.DataFrame(rows)

def ensure_image_embeddings(frame: pd.DataFrame) -> np.ndarray:
    if "image_embeddings" in globals():
        embeddings = globals()["image_embeddings"]
    else:
        image_embeddings_path = EMBEDDING_DIR / "image_embeddings.npy"
        if not image_embeddings_path.exists():
            raise RuntimeError(
                f"image_embeddings is not defined and {image_embeddings_path} does not exist. "
                "Run the `Encode Images and Text with BiomedCLIP` section first."
            )
        embeddings = np.load(image_embeddings_path).astype("float32")
        globals()["image_embeddings"] = embeddings
        print("Loaded cached image embeddings:", image_embeddings_path, embeddings.shape)
    if len(embeddings) != len(frame):
        raise RuntimeError(
            f"Image embedding row count mismatch: embeddings={len(embeddings)} rows={len(frame)}. "
            "Regenerate image embeddings after changing MAX_IMAGES or the split CSV."
        )
    return np.ascontiguousarray(embeddings.astype("float32"))


image_embeddings = ensure_image_embeddings(studies_df)
similar_edges_df = materialize_similarity_edges(studies_df, image_embeddings, KNN_K)
similar_edges_df.to_csv(GRAPH_TABLE_DIR / "kg_edges_faiss_similar_to.csv", index=False)
all_edges_df = pd.concat([edges_df, similar_edges_df], ignore_index=True).fillna("")
all_edges_df.to_csv(GRAPH_TABLE_DIR / "kg_edges_all.csv", index=False)
print("similar_to edges", similar_edges_df.shape)
display(similar_edges_df.head())


Loaded cached image embeddings: /data/liangz2/openi/multi_kg/kg_embeddings/image_embeddings.npy (44100, 512)
similar_to edges (882000, 11)


,src,dst,edge_type,src_type,dst_type,weight,rank,study_id,dst_study_id,source,same_subject
0,image::50003542,image::53006026,similar_to,image,image,0.987070,1,50003542,53006026,faiss_biomedclip,False
1,image::50003542,image::50384171,similar_to,image,image,0.983474,2,50003542,50384171,faiss_biomedclip,False
2,image::50003542,image::52147890,similar_to,image,image,0.982742,3,50003542,52147890,faiss_biomedclip,False
3,image::50003542,image::56793108,similar_to,image,image,0.982491,4,50003542,56793108,faiss_biomedclip,False
4,image::50003542,image::50828606,similar_to,image,image,0.982391,5,50003542,50828606,faiss_biomedclip,False


## Add Node Feature Matrices

This saves type-specific node feature arrays plus node-index maps. These artifacts are designed for a later heterogeneous GraphSAGE/PyG model.


In [10]:
def ensure_image_embeddings_for_features(frame: pd.DataFrame) -> np.ndarray:
    if "image_embeddings" in globals():
        embeddings = globals()["image_embeddings"]
    else:
        image_embeddings_path = EMBEDDING_DIR / "image_embeddings.npy"
        if not image_embeddings_path.exists():
            raise RuntimeError(
                f"image_embeddings is not defined and {image_embeddings_path} does not exist. "
                "Run the `Encode Images and Text with BiomedCLIP` section first."
            )
        embeddings = np.load(image_embeddings_path).astype("float32")
        globals()["image_embeddings"] = embeddings
        print("Loaded cached image embeddings:", image_embeddings_path, embeddings.shape)
    if len(embeddings) != len(frame):
        raise RuntimeError(
            f"Image embedding row count mismatch: embeddings={len(embeddings)} rows={len(frame)}. "
            "Regenerate image embeddings after changing MAX_IMAGES or the split CSV."
        )
    return np.ascontiguousarray(embeddings.astype("float32"))


def ensure_biomedclip_text_encoder() -> None:
    if "encode_texts" in globals():
        return
    try:
        import open_clip
        from open_clip import create_model_from_pretrained, get_tokenizer
    except ImportError as exc:
        raise ImportError("Install open_clip_torch first, or rerun the BiomedCLIP encoding section.") from exc

    model, _ = create_model_from_pretrained(BIOMEDCLIP_REPO)
    tokenizer = get_tokenizer(BIOMEDCLIP_REPO)
    model = model.to(DEVICE, dtype=DTYPE).eval()
    for param in model.parameters():
        param.requires_grad_(False)
    globals()["biomedclip"] = model
    globals()["biomedclip_tokenizer"] = tokenizer

    @torch.no_grad()
    def _encode_texts(texts: Sequence[str], batch_size: int = 64) -> np.ndarray:
        features = []
        for start in tqdm(range(0, len(texts), batch_size), desc="BiomedCLIP text encode"):
            batch = list(texts[start:start + batch_size])
            tokens = tokenizer(batch, context_length=256).to(DEVICE)
            feats = model.encode_text(tokens).float()
            if NORMALIZE_EMBEDDINGS:
                feats = F.normalize(feats, dim=-1)
            features.append(feats.cpu().numpy().astype("float32"))
        return np.ascontiguousarray(np.concatenate(features, axis=0).astype("float32")) if texts else np.empty((0, 512), dtype="float32")

    globals()["encode_texts"] = _encode_texts
    print("Loaded BiomedCLIP text encoder for node features:", BIOMEDCLIP_REPO)


image_embeddings = ensure_image_embeddings_for_features(studies_df)
ensure_biomedclip_text_encoder()

FEATURE_DIR = NODE_FEATURE_DIR
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

node_feature_maps = {}
node_index_tables = {}

# Image features are already encoded and aligned to studies_df.
image_node_ids = [f"image::{sid}" for sid in studies_df["study_id"].astype(str)]
image_index_df = pd.DataFrame({"node_id": image_node_ids, "row_pos": np.arange(len(image_node_ids)), "study_id": studies_df["study_id"].astype(str)})
np.save(FEATURE_DIR / "image_features.npy", image_embeddings.astype("float32"))
image_index_df.to_csv(FEATURE_DIR / "image_nodes.csv", index=False)
node_feature_maps["image"] = image_embeddings.astype("float32")
node_index_tables["image"] = image_index_df

# Text-node features for labels, observations, and anatomy.
for node_type in ["label", "observation", "anatomy"]:
    part = nodes_df[nodes_df["node_type"] == node_type].copy().reset_index(drop=True)
    if part.empty:
        continue
    texts = part["canonical_name"].astype(str).tolist()
    features = encode_texts(texts, batch_size=64)
    part[["node_id", "canonical_name"]].to_csv(FEATURE_DIR / f"{node_type}_nodes.csv", index=False)
    np.save(FEATURE_DIR / f"{node_type}_features.npy", features.astype("float32"))
    node_feature_maps[node_type] = features.astype("float32")
    node_index_tables[node_type] = part[["node_id", "canonical_name"]].copy()
    print(node_type, features.shape)

print("feature artifacts saved to", FEATURE_DIR)


/data/liangz2/diffuser2/lib/python3.12/site-packages/torch/jit/_script.py:1640: DeprecationWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
/data/liangz2/diffuser2/lib/python3.12/site-packages/transformers/models/bert/tokenization_bert.py:104: DeprecationWarning: Deprecated in 0.9.0: WordPiece.__init__ will not create from files anymore, try `WordPiece.from_file` instead
  self._tokenizer = Tokenizer(WordPiece(self._vocab, unk_token=str(unk_token)))


Loaded BiomedCLIP text encoder for node features: hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224


BiomedCLIP text encode:   0%|          | 0/1 [00:00<?, ?it/s]

label (14, 512)


BiomedCLIP text encode:   0%|          | 0/55 [00:00<?, ?it/s]

observation (3492, 512)


BiomedCLIP text encode:   0%|          | 0/33 [00:00<?, ?it/s]

anatomy (2105, 512)
feature artifacts saved to /data/liangz2/openi/multi_kg/kg_node_features


## Build a NetworkX Graph Artifact

This NetworkX graph is useful for debugging, visualization, and GraphRAG-style subgraph retrieval. Large feature arrays are kept outside the graph file and referenced by node IDs.


In [11]:
import networkx as nx

G = nx.MultiDiGraph()
for _, row in nodes_df.iterrows():
    attrs = {k: v for k, v in row.to_dict().items() if k != "node_id"}
    G.add_node(row["node_id"], **attrs)

for _, row in all_edges_df.iterrows():
    attrs = {k: v for k, v in row.to_dict().items() if k not in {"src", "dst"}}
    G.add_edge(row["src"], row["dst"], key=row.get("edge_type", "edge"), **attrs)

nx.write_graphml(G, NETWORKX_DIR / "kg_networkx.graphml")
print(G)
print("GraphML:", NETWORKX_DIR / "kg_networkx.graphml")


MultiDiGraph with 49711 nodes and 1419276 edges
GraphML: /data/liangz2/openi/multi_kg/kg_networkx/kg_networkx.graphml


## GraphRAG-Style Query Subgraph Retrieval

For a query image, the query node's `has_finding` edges should be hidden. The retrieved subgraph contains the query image node, FAISS neighbors, their label/report KG neighborhoods, shared label nodes, and concept nodes connected through RadGraph and lexicon edges.


In [12]:
def retrieve_neighbors_for_embedding(query_embedding: np.ndarray, gallery_embeddings: np.ndarray, top_k: int = QUERY_TOP_K) -> Tuple[np.ndarray, np.ndarray]:
    index = build_faiss_index(gallery_embeddings)
    sims, idx = index.search(l2_normalize_np(query_embedding.reshape(1, -1).astype("float32")), min(top_k, len(gallery_embeddings)))
    return sims[0], idx[0]


def expand_image_neighborhood(graph: nx.MultiDiGraph, image_node: str, hops: int = 2) -> set:
    nodes = {image_node}
    frontier = {image_node}
    for _ in range(hops):
        next_frontier = set()
        for node in frontier:
            next_frontier.update(graph.successors(node))
            next_frontier.update(graph.predecessors(node))
        next_frontier -= nodes
        nodes |= next_frontier
        frontier = next_frontier
    return nodes


def build_graphrag_query_subgraph(
    query_image_path: Optional[str] = None,
    query_row_pos: int = 0,
    top_k: int = QUERY_TOP_K,
    hide_query_labels: bool = True,
    output_prefix: str = "query_subgraph",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if query_image_path is None:
        query_embedding = image_embeddings[query_row_pos]
        query_study_id = str(studies_df.iloc[query_row_pos]["study_id"])
    else:
        query_frame = pd.DataFrame([{"image_path": query_image_path, "study_id": "external_query"}])
        query_embedding = encode_images(query_frame)[0]
        query_study_id = "external_query"

    sims, idx = retrieve_neighbors_for_embedding(query_embedding, image_embeddings, top_k=top_k + 1)
    neighbor_rows = []
    for sim, j in zip(sims, idx):
        j = int(j)
        if j < 0:
            continue
        if query_image_path is None and j == query_row_pos:
            continue
        neighbor_rows.append((j, float(sim)))
        if len(neighbor_rows) >= top_k:
            break

    query_node = f"image::{query_study_id}"
    sub_nodes = {query_node}
    sub_edges = []

    # Add a temporary query node if it does not already exist in the corpus graph.
    query_row = {
        "node_id": query_node, "node_type": "image", "name": query_study_id, "canonical_name": query_study_id,
        "study_id": query_study_id, "image_path": query_image_path or str(studies_df.iloc[query_row_pos]["image_path"]),
        "is_query": True,
    }

    for rank, (j, sim) in enumerate(neighbor_rows, start=1):
        neighbor_study = str(studies_df.iloc[j]["study_id"])
        neighbor_node = f"image::{neighbor_study}"
        sub_nodes.add(neighbor_node)
        sub_edges.append({
            "src": query_node, "dst": neighbor_node, "edge_type": "similar_to", "src_type": "image", "dst_type": "image",
            "weight": sim, "rank": rank, "source": "query_faiss_biomedclip",
        })
        sub_nodes |= expand_image_neighborhood(G, neighbor_node, hops=2)

    # Keep shared label nodes available for dense image-label scoring.
    sub_nodes |= {f"label::{label}" for label in SCORING_LABELS}

    node_records = []
    node_lookup = nodes_df.set_index("node_id").to_dict(orient="index")
    for nid in sorted(sub_nodes):
        if nid == query_node and nid not in node_lookup:
            node_records.append(query_row)
        elif nid in node_lookup:
            row = {"node_id": nid, **node_lookup[nid]}
            row["is_query"] = bool(nid == query_node)
            node_records.append(row)

    sub_node_set = {r["node_id"] for r in node_records}
    for _, edge in all_edges_df.iterrows():
        if edge["src"] in sub_node_set and edge["dst"] in sub_node_set:
            if hide_query_labels and edge["src"] == query_node and edge["edge_type"] == "has_finding":
                continue
            sub_edges.append(edge.to_dict())

    sub_nodes_df = pd.DataFrame(node_records).fillna("")
    sub_edges_df = pd.DataFrame(sub_edges).fillna("")
    sub_nodes_df.to_csv(QUERY_SUBGRAPH_DIR / f"{output_prefix}_nodes.csv", index=False)
    sub_edges_df.to_csv(QUERY_SUBGRAPH_DIR / f"{output_prefix}_edges.csv", index=False)
    return sub_nodes_df, sub_edges_df

query_nodes_df, query_edges_df = build_graphrag_query_subgraph(query_row_pos=0, top_k=QUERY_TOP_K)
print("query subgraph nodes", query_nodes_df.shape, "edges", query_edges_df.shape)
display(query_nodes_df.head())
display(query_edges_df.head())


query subgraph nodes (46828, 9) edges (2131060, 15)


,node_id,node_type,name,canonical_name,study_id,subject_id,image_path,row_pos,is_query
0,anatomy::,anatomy,.,,,,,,False
1,anatomy::/,anatomy,/,/,,,,,False
2,anatomy::0 pulmonary window,anatomy,0 pulmonary window,0 pulmonary window,,,,,False
3,anatomy::2,anatomy,2,2,,,,,False
4,anatomy::2 cm below,anatomy,- 2 cm below,2 cm below,,,,,False


,src,dst,edge_type,src_type,dst_type,weight,rank,source,label,study_id,radgraph_status,surface_text,dst_surface_text,dst_study_id,same_subject
0,image::50003542,image::53006026,similar_to,image,image,0.987070,1,query_faiss_biomedclip,,,,,,,
1,image::50003542,image::50384171,similar_to,image,image,0.983474,2,query_faiss_biomedclip,,,,,,,
2,image::50003542,image::52147890,similar_to,image,image,0.982742,3,query_faiss_biomedclip,,,,,,,
3,image::50003542,image::56793108,similar_to,image,image,0.982491,4,query_faiss_biomedclip,,,,,,,
4,image::50003542,image::50828606,similar_to,image,image,0.982391,5,query_faiss_biomedclip,,,,,,,


## Optional: Export to PyG `HeteroData`

This cell converts the saved node/edge tables into a PyTorch Geometric heterogeneous graph. It is optional because the KG artifact itself is already saved as CSV/GraphML. Reverse edge types are added so messages can flow both ways.


## Generate 10 Fold-Specific KG Files

This section uses `multi_kg_image_json_10fold.csv` fold columns. For each fold, rows marked `train` are used to build the graph and rows marked `test` are saved as query/evaluation metadata. This avoids putting held-out test labels or held-out test image neighborhoods inside the fold training graph.

Each fold directory contains:

- `kg_heterodata_fold{i}.pt`: PyG `HeteroData` if `torch_geometric` is installed, otherwise a compatible dictionary with `x_dict`, `edge_index_dict`, and metadata.
- `kg_nodes_fold{i}.csv`, `kg_edges_all_fold{i}.csv`: inspectable graph tables.
- `train_metadata_fold{i}.csv`: graph image nodes and labels.
- `test_query_metadata_fold{i}.csv`, `test_query_embeddings_fold{i}.npy`: held-out query/evaluation rows.


In [ ]:
# FOLD_KG_DIR is defined in the configuration cell and points to OUTPUT_ROOT / "kg_10fold".
TEXT_FEATURE_CACHE: Dict[str, np.ndarray] = {}


def get_heterodata_class():
    try:
        from torch_geometric.data import HeteroData
        return HeteroData
    except ImportError:
        return None


def encode_texts_cached(texts: Sequence[str]) -> np.ndarray:
    texts = [str(t) for t in texts]
    missing = []
    seen = set()
    for text in texts:
        if text not in TEXT_FEATURE_CACHE and text not in seen:
            missing.append(text)
            seen.add(text)
    if missing:
        features = encode_texts(missing, batch_size=64)
        for text, feat in zip(missing, features):
            TEXT_FEATURE_CACHE[text] = feat.astype("float32")
    if not texts:
        return np.empty((0, image_embeddings.shape[1]), dtype="float32")
    return np.stack([TEXT_FEATURE_CACHE[text] for text in texts], axis=0).astype("float32")


def fold_train_test_frames(frame: pd.DataFrame, fold_index: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    fold_col = f"fold_{fold_index}"
    if fold_col not in frame.columns:
        raise KeyError(f"{fold_col} not found. Check the split CSV columns.")
    values = frame[fold_col].astype(str).str.lower().str.strip()
    train_df = frame[values == "train"].copy().reset_index(drop=True)
    test_df = frame[values == "test"].copy().reset_index(drop=True)
    if train_df.empty or test_df.empty:
        raise RuntimeError(f"Fold {fold_index} has train={len(train_df)} test={len(test_df)}; expected both nonempty.")
    return train_df, test_df


def node_features_for_fold(nodes: pd.DataFrame, train_frame: pd.DataFrame, train_embeddings: np.ndarray):
    node_feature_maps = {}
    node_index_tables = {}

    image_node_ids = [f"image::{sid}" for sid in train_frame["study_id"].astype(str)]
    image_table = pd.DataFrame({
        "node_id": image_node_ids,
        "study_id": train_frame["study_id"].astype(str).to_numpy(),
        "image_path": train_frame["image_path"].astype(str).to_numpy(),
    })
    node_feature_maps["image"] = train_embeddings.astype("float32")
    node_index_tables["image"] = image_table

    for node_type in ["label", "observation", "anatomy", "radgraph_entity"]:
        part = nodes[nodes["node_type"] == node_type].copy().reset_index(drop=True)
        if part.empty:
            continue
        texts = part["canonical_name"].astype(str).tolist()
        node_feature_maps[node_type] = encode_texts_cached(texts)
        node_index_tables[node_type] = part[["node_id", "canonical_name"]].copy()
    return node_feature_maps, node_index_tables


def add_edge_block_to_data(data, is_pyg: bool, edge_type_name: str, src_type: str, dst_type: str, frame: pd.DataFrame, type_to_node_index: Dict[str, Dict[str, int]]):
    src_map = type_to_node_index.get(src_type, {})
    dst_map = type_to_node_index.get(dst_type, {})
    src_idx, dst_idx, weights = [], [], []
    for _, edge in frame.iterrows():
        src, dst = str(edge["src"]), str(edge["dst"])
        if src in src_map and dst in dst_map:
            src_idx.append(src_map[src])
            dst_idx.append(dst_map[dst])
            try:
                weights.append(float(edge.get("weight", 1.0) or 1.0))
            except Exception:
                weights.append(1.0)
    if not src_idx:
        return
    edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)
    edge_weight = torch.tensor(weights, dtype=torch.float32)
    etype = (src_type, edge_type_name, dst_type)
    rev_etype = (dst_type, f"rev_{edge_type_name}", src_type)
    if is_pyg:
        data[etype].edge_index = edge_index
        data[etype].edge_weight = edge_weight
        data[rev_etype].edge_index = edge_index.flip(0)
        data[rev_etype].edge_weight = edge_weight
    else:
        data["edge_index_dict"][etype] = edge_index
        data["edge_weight_dict"][etype] = edge_weight
        data["edge_index_dict"][rev_etype] = edge_index.flip(0)
        data["edge_weight_dict"][rev_etype] = edge_weight


def build_heterodata_object(
    fold_index: int,
    nodes: pd.DataFrame,
    edges: pd.DataFrame,
    train_frame: pd.DataFrame,
    train_embeddings: np.ndarray,
):
    feature_maps, index_tables = node_features_for_fold(nodes, train_frame, train_embeddings)
    HeteroData = get_heterodata_class()
    is_pyg = HeteroData is not None
    data = HeteroData() if is_pyg else {"x_dict": {}, "edge_index_dict": {}, "edge_weight_dict": {}, "node_ids": {}, "y_dict": {}, "metadata": {}}
    type_to_node_index = {}

    for node_type, table in index_tables.items():
        table = table.reset_index(drop=True).copy()
        features = feature_maps[node_type].astype("float32")
        node_ids = table["node_id"].astype(str).tolist()
        type_to_node_index[node_type] = {node_id: i for i, node_id in enumerate(node_ids)}
        if is_pyg:
            data[node_type].x = torch.from_numpy(features).float()
            data[node_type].node_id = node_ids
            if node_type == "image":
                data[node_type].y = torch.from_numpy(train_frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)).float()
                data[node_type].study_id = train_frame["study_id"].astype(str).tolist()
                data[node_type].image_path = train_frame["image_path"].astype(str).tolist()
            if node_type == "label":
                data[node_type].label_name = table["canonical_name"].astype(str).tolist()
        else:
            data["x_dict"][node_type] = torch.from_numpy(features).float()
            data["node_ids"][node_type] = node_ids
            if node_type == "image":
                data["y_dict"][node_type] = torch.from_numpy(train_frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)).float()

    for (src_type, edge_type_name, dst_type), part in edges.groupby(["src_type", "edge_type", "dst_type"]):
        add_edge_block_to_data(data, is_pyg, str(edge_type_name), str(src_type), str(dst_type), part, type_to_node_index)

    metadata = {
        "fold": int(fold_index),
        "is_pyg_heterodata": bool(is_pyg),
        "node_types": sorted(index_tables.keys()),
        "target_labels": list(SCORING_LABELS),
        "target_columns": list(TARGET_COLUMNS),
        "num_train_images": int(len(train_frame)),
    }
    if is_pyg:
        data.fold = int(fold_index)
        data.target_labels = list(SCORING_LABELS)
        data.target_columns = list(TARGET_COLUMNS)
    else:
        data["metadata"].update(metadata)
    return data, metadata


def build_one_fold_kg(fold_index: int) -> Dict:
    fold_dir = FOLD_KG_DIR / f"fold{fold_index}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    kg_path = fold_dir / f"kg_heterodata_fold{fold_index}.pt"
    manifest_path = fold_dir / f"kg_manifest_fold{fold_index}.json"
    if REUSE_EXISTING_FOLD_KG and kg_path.exists() and manifest_path.exists():
        print(f"Fold {fold_index}: existing KG found, skipping {kg_path}")
        with open(manifest_path, "r", encoding="utf-8") as f:
            return json.load(f)

    train_df, test_df = fold_train_test_frames(studies_df, fold_index)
    train_pos = train_df["global_row_pos"].to_numpy(dtype=np.int64)
    test_pos = test_df["global_row_pos"].to_numpy(dtype=np.int64)
    train_embeddings = image_embeddings[train_pos].astype("float32")
    test_embeddings = image_embeddings[test_pos].astype("float32")

    train_df.to_csv(fold_dir / f"train_metadata_fold{fold_index}.csv", index=False)
    test_df.to_csv(fold_dir / f"test_query_metadata_fold{fold_index}.csv", index=False)
    np.save(fold_dir / f"train_image_embeddings_fold{fold_index}.npy", train_embeddings)
    np.save(fold_dir / f"test_query_embeddings_fold{fold_index}.npy", test_embeddings)

    fold_nodes, fold_rad_edges = build_kg_tables(train_df, radgraph_annotations)
    fold_sim_edges = materialize_similarity_edges(train_df, train_embeddings, KNN_K)
    fold_edges = pd.concat([fold_rad_edges, fold_sim_edges], ignore_index=True).fillna("")

    fold_nodes.to_csv(fold_dir / f"kg_nodes_fold{fold_index}.csv", index=False)
    fold_rad_edges.to_csv(fold_dir / f"kg_edges_radgraph_and_labels_fold{fold_index}.csv", index=False)
    fold_sim_edges.to_csv(fold_dir / f"kg_edges_faiss_similar_to_fold{fold_index}.csv", index=False)
    fold_edges.to_csv(fold_dir / f"kg_edges_all_fold{fold_index}.csv", index=False)

    data, hetero_meta = build_heterodata_object(fold_index, fold_nodes, fold_edges, train_df, train_embeddings)
    torch.save(data, kg_path)

    manifest = {
        "fold": int(fold_index),
        "kg_path": str(kg_path),
        "fold_dir": str(fold_dir),
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "num_nodes": int(len(fold_nodes)),
        "num_edges": int(len(fold_edges)),
        "node_type_counts": fold_nodes["node_type"].value_counts().to_dict(),
        "edge_type_counts": fold_edges["edge_type"].value_counts().to_dict(),
        **hetero_meta,
    }
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Fold {fold_index}: saved {kg_path}")
    return manifest


fold_kg_manifests = []
if BUILD_10FOLD_KG_FILES:
    missing_fold_cols = [f"fold_{i}" for i in FOLD_INDICES if f"fold_{i}" not in studies_df.columns]
    if missing_fold_cols:
        print("Skipping 10-fold KG generation because split columns are missing:", missing_fold_cols)
    else:
        for fold_index in FOLD_INDICES:
            fold_kg_manifests.append(build_one_fold_kg(fold_index))
        fold_manifest_df = pd.DataFrame(fold_kg_manifests)
        fold_manifest_df.to_csv(FOLD_KG_DIR / "fold_kg_manifest_summary.csv", index=False)
        display(fold_manifest_df[["fold", "train_rows", "test_rows", "num_nodes", "num_edges", "kg_path"]])


BiomedCLIP text encode:   0%|          | 0/1 [00:00<?, ?it/s]

BiomedCLIP text encode:   0%|          | 0/52 [00:00<?, ?it/s]

BiomedCLIP text encode:   0%|          | 0/15 [00:00<?, ?it/s]

/data/liangz2/diffuser2/lib/python3.12/site-packages/torch_geometric/llm/utils/backend_utils.py:26: DeprecationWarning: `torch_geometric.distributed` has been deprecated since 2.7.0 and will no longer be maintained. For distributed training, refer to our tutorials on distributed training at https://pytorch-geometric.readthedocs.io/en/latest/tutorial/distributed.html or cuGraph examples at https://github.com/rapidsai/cugraph-gnn/tree/main/python/cugraph-pyg/cugraph_pyg/examples
  from torch_geometric.distributed import (
/data/liangz2/diffuser2/lib/python3.12/site-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/tmp/ipykernel_4097150/3781400265.py:118: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or mak

Fold 0: saved /data/liangz2/openi/multi_kg/kg_10fold/fold0/kg_heterodata_fold0.pt


BiomedCLIP text encode:   0%|          | 0/3 [00:00<?, ?it/s]

BiomedCLIP text encode:   0%|          | 0/1 [00:00<?, ?it/s]

Fold 1: saved /data/liangz2/openi/multi_kg/kg_10fold/fold1/kg_heterodata_fold1.pt
Fold 2: saved /data/liangz2/openi/multi_kg/kg_10fold/fold2/kg_heterodata_fold2.pt
Fold 3: saved /data/liangz2/openi/multi_kg/kg_10fold/fold3/kg_heterodata_fold3.pt
Fold 4: saved /data/liangz2/openi/multi_kg/kg_10fold/fold4/kg_heterodata_fold4.pt
Fold 5: saved /data/liangz2/openi/multi_kg/kg_10fold/fold5/kg_heterodata_fold5.pt
Fold 6: saved /data/liangz2/openi/multi_kg/kg_10fold/fold6/kg_heterodata_fold6.pt


In [ ]:
BUILD_PYG_HETERODATA = False

if BUILD_PYG_HETERODATA:
    try:
        from torch_geometric.data import HeteroData
    except ImportError as exc:
        raise ImportError("Install torch_geometric before enabling BUILD_PYG_HETERODATA.") from exc

    data = HeteroData()
    type_to_node_index = {}

    for node_type, table in node_index_tables.items():
        features = node_feature_maps[node_type]
        table = table.reset_index(drop=True).copy()
        type_to_node_index[node_type] = {nid: i for i, nid in enumerate(table["node_id"].astype(str))}
        data[node_type].x = torch.from_numpy(features).float()
        data[node_type].node_id = list(table["node_id"].astype(str))

    def add_edge_block(edge_type_name: str, src_type: str, dst_type: str, frame: pd.DataFrame, weight_col: str = "weight"):
        src_map = type_to_node_index.get(src_type, {})
        dst_map = type_to_node_index.get(dst_type, {})
        src_idx, dst_idx, weights = [], [], []
        for _, edge in frame.iterrows():
            s, d = str(edge["src"]), str(edge["dst"])
            if s in src_map and d in dst_map:
                src_idx.append(src_map[s])
                dst_idx.append(dst_map[d])
                weights.append(float(edge.get(weight_col, 1.0) or 1.0))
        if not src_idx:
            return
        edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)
        data[(src_type, edge_type_name, dst_type)].edge_index = edge_index
        data[(src_type, edge_type_name, dst_type)].edge_weight = torch.tensor(weights, dtype=torch.float32)
        data[(dst_type, f"rev_{edge_type_name}", src_type)].edge_index = edge_index.flip(0)
        data[(dst_type, f"rev_{edge_type_name}", src_type)].edge_weight = torch.tensor(weights, dtype=torch.float32)

    for (src_type, edge_type_name, dst_type), part in all_edges_df.groupby(["src_type", "edge_type", "dst_type"]):
        add_edge_block(str(edge_type_name), str(src_type), str(dst_type), part)

    torch.save(data, PYG_DIR / "kg_heterodata.pt")
    print(data)
    print("Saved", PYG_DIR / "kg_heterodata.pt")


## Artifact Summary

The main files written by this notebook are:

- `kg_metadata/study_subset_metadata.csv`: selected image/JSON records and 14-label targets, including `fold_0` ... `fold_9` when loaded from the split CSV.
- `kg_embeddings/image_embeddings.npy`: BiomedCLIP image embeddings for all loaded rows.
- `kg_radgraph/radgraph_raw_annotations.json`: cached RadGraph output.
- `kg_graph_tables/kg_nodes.csv`: image, label, observation, and anatomy nodes for the current loaded corpus/subset.
- `kg_graph_tables/kg_edges_radgraph_and_labels.csv`: JSON labels plus RadGraph-derived mention/relation/map edges.
- `kg_graph_tables/kg_edges_faiss_similar_to.csv`: FAISS image-neighbor edges.
- `kg_graph_tables/kg_edges_all.csv`: merged edge table.
- `kg_node_features/`: type-specific feature arrays and node-index maps.
- `kg_networkx/kg_networkx.graphml`: graph artifact for inspection.
- `kg_query_subgraphs/query_subgraph_nodes.csv` and `kg_query_subgraphs/query_subgraph_edges.csv`: GraphRAG-style retrieved subgraph for one query image.
- `kg_10fold/fold{i}/kg_heterodata_fold{i}.pt`: train-only heterogeneous graph file for fold `i`.
- `kg_10fold/fold{i}/test_query_metadata_fold{i}.csv` and `test_query_embeddings_fold{i}.npy`: held-out query/evaluation rows for fold `i`.

For full-scale 10-fold experiments, use the fold-specific `kg_heterodata_fold{i}.pt` files for training and the corresponding test query files for inductive evaluation. The fold graph contains only train rows, so the query image's `has_finding` labels remain hidden at inference time.
